# IDEAL archetype pipeline notebook

This notebook is the cleaned starting point for the revised project design:

- Main analysis: the 39 enhanced IDEAL homes.
- Extension if time allows: compare whole-home electricity archetype shapes against the wider 255-home sample.

The notebook is written to be safe on the current local repository structure. It uses relative paths and auto-detects the project root, so it does not depend on old Windows paths.

Current scope of this notebook:

1. Confirm the enhanced 39-home electricity and gas files can be found.
2. Load metadata from the local `metadata_and_surveys (1)/metadata` folder.
3. Build a home-level metadata matrix for later modelling.
4. Build 2-hour whole-home electricity and gas feature tables for the 39 enhanced homes.
5. Save clean CSV outputs that can be used for clustering and later report figures.

This notebook does not yet run clustering. It is the Stage 0 to Stage 1 data-preparation notebook for the `39 homes main analysis + 255 homes extension` strategy.

## Step 1. Set paths and helper functions

This first code cell only does notebook setup:

- import packages
- locate the IDEAL project root automatically
- define helper functions for reading metadata and sensor files
- define reusable functions for calendar features and 2-hour aggregation

Run this cell first. The rest of the notebook depends on these helper functions.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 20)


def locate_project_root(start: Path | None = None) -> Path:
    """Walk upwards until the IDEAL project root is found."""
    start = (start or Path.cwd()).resolve()
    markers = ["sensordata", "script", "metadata_and_surveys (1)"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError(
        "Could not locate the IDEAL project root from the current working directory."
    )


def read_csv_flexible(path: Path, **kwargs) -> pd.DataFrame:
    """Try common encodings used by the IDEAL metadata exports."""
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def extract_homeid(path: Path) -> int:
    match = re.search(r"home(\d+)_", path.name)
    if not match:
        raise ValueError(f"Could not parse homeid from file name: {path.name}")
    return int(match.group(1))


def read_sensor_series(path: Path, value_name: str) -> pd.DataFrame:
    """Read IDEAL sensor files stored as timestamp,value without a normal header row."""
    df = pd.read_csv(
        path,
        compression="infer",
        header=None,
        names=["time", value_name],
        usecols=[0, 1],
    )
    df["homeid"] = extract_homeid(path)
    df["source_file"] = path.name
    return df


def preview_sensor_group(file_paths: list[Path], value_name: str, n_files: int = 2, n_rows: int = 5) -> pd.DataFrame:
    previews = []
    for path in file_paths[:n_files]:
        preview = read_sensor_series(path, value_name=value_name).head(n_rows)
        previews.append(preview)
    if not previews:
        return pd.DataFrame(columns=["time", value_name, "homeid", "source_file"])
    return pd.concat(previews, ignore_index=True)


def season_from_month(month: int) -> str:
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    if month in [9, 10, 11]:
        return "Autumn"
    return "Winter"


def add_calendar_features(df: pd.DataFrame, time_col: str = "time") -> pd.DataFrame:
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
    out = out.dropna(subset=[time_col])
    out["is_weekend"] = out[time_col].dt.weekday >= 5
    out["day_type"] = np.where(out["is_weekend"], "weekend", "weekday")
    out["season"] = out[time_col].dt.month.map(season_from_month)
    out["current_hour"] = out[time_col].dt.hour
    out["date"] = out[time_col].dt.date
    return out


def build_two_hour_features_from_files(file_paths: list[Path], value_col: str, feature_matrix: pd.DataFrame, agg: str = "sum", progress_every: int = 5) -> pd.DataFrame:
    feature_lookup = feature_matrix.copy()
    if "homeid" in feature_lookup.columns:
        feature_lookup["homeid"] = feature_lookup["homeid"].astype(int)

    aggregated_frames = []
    for i, path in enumerate(file_paths, start=1):
        homeid = extract_homeid(path)
        df = read_sensor_series(path, value_name=value_col)
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
        df = df.dropna(subset=["time", value_col])
        df["homeid"] = homeid

        indexed = df.set_index("time")
        two_hour = indexed.resample("2h")[value_col].agg(agg).reset_index()
        two_hour["homeid"] = homeid

        daily_peak = (
            indexed.resample("1D")[value_col]
            .apply(lambda x: x.idxmax().hour if not x.isnull().all() else np.nan)
            .reset_index()
            .rename(columns={value_col: "day_peak_hour", "time": "peak_date"})
        )
        daily_peak["peak_date"] = daily_peak["peak_date"].dt.date
        daily_peak["homeid"] = homeid

        two_hour = add_calendar_features(two_hour, time_col="time")
        enriched = two_hour.merge(
            feature_lookup[feature_lookup["homeid"] == homeid],
            on="homeid",
            how="left",
        )
        enriched = enriched.merge(
            daily_peak,
            left_on=["homeid", "date"],
            right_on=["homeid", "peak_date"],
            how="left",
        )
        enriched = enriched.drop(columns=["peak_date"])
        aggregated_frames.append(enriched)

        del df, indexed, two_hour, daily_peak, enriched

        if i % progress_every == 0 or i == len(file_paths):
            print(f"Processed {i}/{len(file_paths)} files for {value_col}")

    if not aggregated_frames:
        return pd.DataFrame()
    return pd.concat(aggregated_frames, ignore_index=True).sort_values(["homeid", "time"]).reset_index(drop=True)


ROOT = locate_project_root()
SENSORDATA_DIR = ROOT / "sensordata"
ROOM_APPLIANCE_DIR = ROOT / "room_and_appliance_sensors" / "sensordata"
METADATA_DIR = ROOT / "metadata_and_surveys (1)" / "metadata"
SCRIPT_DIR = ROOT / "script"

print(f"Project root: {ROOT}")
print(f"Sensor data: {SENSORDATA_DIR}")
print(f"Room/appliance sensor data: {ROOM_APPLIANCE_DIR}")
print(f"Metadata: {METADATA_DIR}")

## Step 2. Define the enhanced 39-home sample

This cell fixes the enhanced subgroup explicitly so the notebook is reproducible.

It also checks whether all expected electricity and gas files exist locally for those 39 homes.

In [ ]:
# Reproducible enhanced-home sample used in the original 39-home extraction notebook.
ENHANCED_HOME_IDS = [
    61, 62, 63, 65, 73, 90, 96, 105, 106, 128, 136, 139, 140, 145, 146,
    162, 168, 169, 171, 175, 208, 212, 225, 227, 228, 231, 238, 242, 249,
    255, 259, 262, 263, 264, 266, 268, 276, 311, 328,
]

enhanced_electricity_files = sorted(
    path
    for path in SENSORDATA_DIR.glob("*electric-mains_electric-combined.csv.gz")
    if extract_homeid(path) in ENHANCED_HOME_IDS
)

enhanced_gas_files = sorted(
    path
    for path in SENSORDATA_DIR.glob("*gas-pulse_gas.csv.gz")
    if extract_homeid(path) in ENHANCED_HOME_IDS
)

enhanced_electricity_homeids = sorted({extract_homeid(path) for path in enhanced_electricity_files})
enhanced_gas_homeids = sorted({extract_homeid(path) for path in enhanced_gas_files})

print(f"Expected enhanced home count: {len(ENHANCED_HOME_IDS)}")
print(f"Electricity files found: {len(enhanced_electricity_files)} for {len(enhanced_electricity_homeids)} homes")
print(f"Gas files found: {len(enhanced_gas_files)} for {len(enhanced_gas_homeids)} homes")
print(f"Missing electricity homes: {sorted(set(ENHANCED_HOME_IDS) - set(enhanced_electricity_homeids))}")
print(f"Missing gas homes: {sorted(set(ENHANCED_HOME_IDS) - set(enhanced_gas_homeids))}")

## Step 3. Load metadata safely

This cell loads the local metadata tables from `metadata_and_surveys (1)/metadata`.

The loader is intentionally defensive:

- it uses flexible encodings
- it skips files that are not present
- it keeps the notebook from failing because of small export differences

In [ ]:
requested_tables = [
    "home",
    "person",
    "appliance",
    "other_appliance",
    "tariff",
    "weatherfeed",
    "location",
]

dfs = {}
for table_name in requested_tables:
    table_path = METADATA_DIR / f"{table_name}.csv"
    if table_path.exists():
        dfs[table_name] = read_csv_flexible(table_path)
        print(f"Loaded {table_name}.csv with shape {dfs[table_name].shape}")
    else:
        print(f"Skipped missing metadata table: {table_path.name}")

sorted(dfs.keys())

## Step 4. Build the home-level feature matrix

This cell creates one row per enhanced home by combining:

- household metadata from `home.csv`
- working status from `person.csv`
- tariff information
- appliance counts and other appliance counts

This table is the home-level feature block that will later be joined onto the time-series aggregates.

In [ ]:
def available_columns(df: pd.DataFrame, wanted: list[str]) -> list[str]:
    return [col for col in wanted if col in df.columns]


home_cols = available_columns(
    dfs["home"],
    [
        "homeid",
        "residents",
        "income_band",
        "hometype",
        "occupancy",
        "occupied_days",
        "location",
        "build_era",
        "urban_rural_class",
    ],
)
home_feat = dfs["home"][home_cols].copy()

person_cols = available_columns(
    dfs["person"],
    ["homeid", "primaryparticipant", "workingstatus"],
)
person_feat = dfs["person"][person_cols].copy()
if "primaryparticipant" in person_feat.columns:
    person_feat = person_feat[person_feat["primaryparticipant"] == 1]
if "primaryparticipant" in person_feat.columns:
    person_feat = person_feat.drop(columns=["primaryparticipant"])

tariff_cols = available_columns(
    dfs["tariff"],
    ["homeid", "daily_standing_charge_pence", "unit_charge_pence_per_kwh"],
)
tariff_feat = dfs["tariff"][tariff_cols].drop_duplicates("homeid") if tariff_cols else pd.DataFrame(columns=["homeid"])

appliance_df = dfs["appliance"].copy()
appliance_df["number"] = pd.to_numeric(appliance_df.get("number"), errors="coerce").fillna(0)
app_type_features = (
    appliance_df.pivot_table(
        index="homeid",
        columns="appliancetype",
        values="number",
        aggfunc="sum",
        fill_value=0,
    )
    .add_prefix("app_")
    .reset_index()
)
if len(app_type_features.columns) > 1:
    app_value_cols = [col for col in app_type_features.columns if col.startswith("app_")]
    app_type_features["major_app_total"] = app_type_features[app_value_cols].sum(axis=1)

other_appliance_df = dfs["other_appliance"].copy()
other_appliance_df["number"] = pd.to_numeric(other_appliance_df.get("number"), errors="coerce").fillna(0)
other_name_col = "appliance_name" if "appliance_name" in other_appliance_df.columns else "appliancename"
other_app_type_features = (
    other_appliance_df.pivot_table(
        index="homeid",
        columns=other_name_col,
        values="number",
        aggfunc="sum",
        fill_value=0,
    )
    .add_prefix("other_")
    .reset_index()
)
if len(other_app_type_features.columns) > 1:
    other_value_cols = [col for col in other_app_type_features.columns if col.startswith("other_")]
    other_app_type_features["other_app_total"] = other_app_type_features[other_value_cols].sum(axis=1)

feature_matrix = (
    home_feat.merge(person_feat, on="homeid", how="left")
    .merge(tariff_feat, on="homeid", how="left")
    .merge(app_type_features, on="homeid", how="left")
    .merge(other_app_type_features, on="homeid", how="left")
)

count_cols = [col for col in feature_matrix.columns if col.startswith("app_") or col.startswith("other_")]
feature_matrix[count_cols] = feature_matrix[count_cols].fillna(0)
feature_matrix = feature_matrix[feature_matrix["homeid"].isin(ENHANCED_HOME_IDS)].sort_values("homeid").reset_index(drop=True)

print(f"Feature matrix shape: {feature_matrix.shape}")
feature_matrix.head()

## Step 5. Load the 39-home sensor files

This step now does a lightweight preview only.

The reason for keeping this step light is practical: loading all 39 raw files into memory at once can kill the Jupyter kernel.

Important note for interpretation:

- The raw values are kept exactly as stored in the files at this stage.
- This notebook labels the columns as `electricity_value_raw` and `gas_value_raw` on purpose.
- Electricity is treated as power in watts and is later aggregated to 2-hour mean power.
- If you later confirm a unit conversion is needed, apply it before the final clustering stage.

This avoids silently claiming a unit that has not yet been validated.

The heavy aggregation is done in Step 6 file-by-file instead of loading the whole raw dataset into one dataframe.

In [ ]:
electricity_preview = preview_sensor_group(enhanced_electricity_files, value_name="electricity_value_raw")
gas_preview = preview_sensor_group(enhanced_gas_files, value_name="gas_value_raw")

print(f"Enhanced electricity files ready: {len(enhanced_electricity_files)}")
print(f"Enhanced gas files ready: {len(enhanced_gas_files)}")
print("Preview only: Step 5 no longer loads the full raw data into memory.")

display(electricity_preview)
display(gas_preview)

## Step 6. Aggregate to 2-hour features

This cell converts the raw timestamp-level sensor files into cleaner 2-hour feature tables.

To keep memory usage under control, it processes one file at a time and only keeps the aggregated result.

For each fuel it adds:

- 2-hour summed values
- weekday or weekend label
- season
- hour-of-day
- daily peak hour
- merged home-level metadata

In [ ]:
all_electricity_with_features = build_two_hour_features_from_files(
    file_paths=enhanced_electricity_files,
    value_col="electricity_value_raw",
    feature_matrix=feature_matrix,
    agg="mean",
)

all_gas_with_features = build_two_hour_features_from_files(
    file_paths=enhanced_gas_files,
    value_col="gas_value_raw",
    feature_matrix=feature_matrix,
)

all_electricity_with_features = all_electricity_with_features.rename(columns={"electricity_value_raw": "mean_power_w"})

print(f"Electricity 2-hour feature table: {all_electricity_with_features.shape}")
print(f"Gas 2-hour feature table: {all_gas_with_features.shape}")

display(all_electricity_with_features.head())
display(all_gas_with_features.head())

## Step 7. Save cleaned outputs

This cell writes the current stage outputs into the local `script/` folder.

These CSVs are intended to be the stable handoff files for the next notebook stage.

In [ ]:
electricity_output_path = SCRIPT_DIR / "outputs" / "stage1_preparation" / "all_electricity_features_2hr_enhanced.csv"
gas_output_path = SCRIPT_DIR / "outputs" / "stage1_preparation" / "all_gas_features_2hr_enhanced.csv"
metadata_output_path = SCRIPT_DIR / "outputs" / "stage1_preparation" / "enhanced_home_feature_matrix.csv"

all_electricity_with_features.to_csv(electricity_output_path, index=False)
all_gas_with_features.to_csv(gas_output_path, index=False)
feature_matrix.to_csv(metadata_output_path, index=False)

print(f"Saved electricity features to: {electricity_output_path}")
print(f"Saved gas features to: {gas_output_path}")
print(f"Saved home feature matrix to: {metadata_output_path}")

## Step 8. Build a home-day electricity table

This step converts the 2-hour electricity table into one row per `homeid + date`.

The output keeps:

- the 12 daily 2-hour blocks as separate columns
- day-level summary statistics
- calendar labels such as weekday or weekend and season
- repeated household metadata for later clustering interpretation

This is the correct level for the next stage, because clustering should be done on daily profiles rather than directly on the long 2-hour table.

In [ ]:
BLOCK_HOURS = list(range(0, 24, 2))
BLOCK_COLS = [f"block_{hour:02d}" for hour in BLOCK_HOURS]
FLAT_THRESHOLD_W = 100

electricity_daily_long = all_electricity_with_features.copy()
electricity_daily_long["date"] = pd.to_datetime(electricity_daily_long["date"], errors="coerce")

block_profiles = (
    electricity_daily_long.pivot_table(
        index=["homeid", "date"],
        columns="current_hour",
        values="mean_power_w",
        aggfunc="mean",
    )
    .reindex(columns=BLOCK_HOURS)
    .rename(columns={hour: f"block_{hour:02d}" for hour in BLOCK_HOURS})
    .reset_index()
)

daily_summary = (
    electricity_daily_long.groupby(["homeid", "date"]).agg(
        block_count=("current_hour", "nunique"),
        day_peak_hour=("day_peak_hour", "first"),
        is_weekend=("is_weekend", "first"),
        day_type=("day_type", "first"),
        season=("season", "first"),
    )
    .reset_index()
)

daytime_mask = electricity_daily_long["current_hour"].between(8, 20)
daytime_total = (
    electricity_daily_long.loc[daytime_mask]
    .groupby(["homeid", "date"])["mean_power_w"]
    .mean()
    .rename("daytime_mean_power_w")
    .reset_index()
)

daily_electricity_features = daily_summary.merge(block_profiles, on=["homeid", "date"], how="left")
daily_electricity_features = daily_electricity_features.merge(daytime_total, on=["homeid", "date"], how="left")
daily_electricity_features["daytime_mean_power_w"] = daily_electricity_features["daytime_mean_power_w"].fillna(0)
daily_electricity_features["daily_total_kwh"] = daily_electricity_features[BLOCK_COLS].sum(axis=1) * 2 / 1000
daily_electricity_features["daily_min_w"] = daily_electricity_features[BLOCK_COLS].min(axis=1)
daily_electricity_features["daily_max_w"] = daily_electricity_features[BLOCK_COLS].max(axis=1)
daily_electricity_features["daily_range_w"] = daily_electricity_features["daily_max_w"] - daily_electricity_features["daily_min_w"]
daily_electricity_features["daytime_fraction"] = np.where(
    daily_electricity_features[BLOCK_COLS].sum(axis=1) > 0,
    daily_electricity_features[[f"block_{hour:02d}" for hour in range(8, 22, 2)]].sum(axis=1) / daily_electricity_features[BLOCK_COLS].sum(axis=1),
    np.nan,
)
daily_electricity_features["complete_day"] = daily_electricity_features["block_count"] == 12
daily_electricity_features["flat_day_flag"] = daily_electricity_features["complete_day"] & (daily_electricity_features["daily_range_w"] < FLAT_THRESHOLD_W)
daily_electricity_features["cluster_ready_day"] = False
daily_electricity_features["day_peak_hour"] = (
    daily_electricity_features[BLOCK_COLS]
    .idxmax(axis=1)
    .str.replace("block_", "", regex=False)
    .astype(float)
)

daily_electricity_features = daily_electricity_features.merge(feature_matrix, on="homeid", how="left")
daily_electricity_features = daily_electricity_features.sort_values(["homeid", "date"]).reset_index(drop=True)

print(f"Daily electricity feature table shape: {daily_electricity_features.shape}")
display(daily_electricity_features.head())

## Step 9. Create profile normalisation inputs

This step prepares the daily profile columns for clustering.

Two important design choices are made here:

- only complete days with all 12 two-hour blocks are marked as ready for clustering
- profile normalisation is based on daily min-max scaling so the shape of demand is emphasised rather than absolute magnitude

Because the electricity unit is still labelled as raw rather than confirmed `kWh`, the flat-day threshold is left as a user-controlled parameter instead of being hard-coded.

In [ ]:
daily_electricity_features["normalisable_day"] = (
    daily_electricity_features["complete_day"]
    & (~daily_electricity_features["flat_day_flag"])
    & daily_electricity_features["daily_range_w"].gt(0)
)

block_min = daily_electricity_features[BLOCK_COLS].min(axis=1)
block_range = daily_electricity_features[BLOCK_COLS].max(axis=1) - block_min

norm_cols = [f"norm_{col}" for col in BLOCK_COLS]
daily_electricity_features[norm_cols] = np.nan

valid_norm = daily_electricity_features["normalisable_day"] & block_range.gt(0)
daily_electricity_features.loc[valid_norm, norm_cols] = (
    daily_electricity_features.loc[valid_norm, BLOCK_COLS]
    .sub(block_min[valid_norm], axis=0)
    .div(block_range[valid_norm], axis=0)
    .to_numpy()
)

daily_electricity_features["cluster_ready_day"] = (
    daily_electricity_features["normalisable_day"]
    & daily_electricity_features[norm_cols].notna().all(axis=1)
)

print("Normalisation complete.")
print(daily_electricity_features[["homeid", "date", "complete_day", "normalisable_day", "cluster_ready_day", "flat_day_flag"]].head().to_string(index=False))

## Step 10. Save daily clustering inputs

This cell saves the daily electricity table that will be used for the next notebook stage.

That next stage should focus on:

- filtering cluster-ready days
- setting or validating the flat-day rule
- selecting the profile columns used in clustering
- running silhouette or elbow checks and then clustering

In [ ]:
daily_output_path = SCRIPT_DIR / "outputs" / "stage1_preparation" / "daily_electricity_features_enhanced.csv"
daily_electricity_features.to_csv(daily_output_path, index=False)
print(f"Saved daily electricity features to: {daily_output_path}")

## Step 11. Optional extension: whole-home comparison against the wider sample

Use the cell below only after the 39-home pipeline is stable.

Recommended use:

- Keep the 39-home outputs as the main dissertation analysis.
- Use the 255-home extension only to check whether broad whole-home daily shapes look consistent.
- Do not mix the framing. The 39-home analysis is the main high-resolution study. The 255-home run is a robustness or comparison layer.

In [ ]:
RUN_255_EXTENSION = False

if RUN_255_EXTENSION:
    full_electricity_files = sorted(SENSORDATA_DIR.glob("*electric-mains_electric-combined.csv.gz"))
    full_feature_matrix = dfs["home"][[col for col in feature_matrix.columns if col in dfs["home"].columns or col == "homeid"]].copy()
    full_feature_matrix = full_feature_matrix.drop_duplicates("homeid")
    full_electricity_two_hour = build_two_hour_features_from_files(
        file_paths=full_electricity_files,
        value_col="electricity_value_raw",
        feature_matrix=full_feature_matrix,
        progress_every=20,
    )
    full_output_path = SCRIPT_DIR / "all_electricity_features_2hr_full_sample.csv"
    full_electricity_two_hour.to_csv(full_output_path, index=False)
    print(f"Saved full-sample electricity features to: {full_output_path}")
else:
    print("255-home extension is disabled. Leave RUN_255_EXTENSION = False until the 39-home pipeline is validated.")